EDA draft

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
DATA_FOLDER: str = "./data/"
UNICEF_DATA_CSV: str = DATA_FOLDER + "unicef_malawi.csv"
OUTPUT_FOLDER: str = "./output/"

In [2]:
unicef_df = pd.read_csv(UNICEF_DATA_CSV)

# Make target binary
Use map to transform target column "FCF26" to a binary column "FCF26_binary", where 0 = not depressed, and 1 = depressed

In [12]:
# depression_map maps values in FCF26 to binary; where 0 = not depressed, and 1 = depressed.
depression_map: dict[str: int] = {
    "NEVER": 0,
    "A FEW TIMES A YEAR": 0,
    "MONTHLY": 0,
    "WEEKLY": 1,
    "DAILY": 1,
}

unicef_df["FCF26_binary"] = unicef_df["FCF26"].map(depression_map)

# Rank correlations to target
We use Cramér's V to measure the association between each feature and the target, removing features with Cramér's V < 0.05, corresponding to less than 0.25% of variance explained individually. While individual features explain modest variance; reflecting the multifactorial and self-reported nature of the data, the combined predictive power of retained features is expected to be substantially higher, as the model learns from many weak signals jointly.

In [40]:
from scipy.stats import chi2_contingency
import numpy as np

def perform_cramers_v(
    *,
    df: pd.DataFrame,
    min_score: float = 0.05,
    target_column_name: str,
    ignore_colums: list,
) -> pd.DataFrame:
    """Performs cramers v test on the columns in the provided 'df' 
    against the 'target_column_name' column and returns a dataframe
    containing the signficiant features that meet the 'min_score'
    threshold. Ignores the 'ignore_colums'.
    """
    results = []
    df_without_target = df.drop(columns=ignore_colums)
    for col in df_without_target.columns:
        contingency_table = pd.crosstab(df[col], df[target_column_name])
        chi2, p, _, _ = chi2_contingency(contingency_table)
        n = contingency_table.sum().sum()
        min_dim = min(contingency_table.shape) - 1
        cramers_v = np.sqrt(chi2 / (n * min_dim))
        results.append({"feature_name": col, "p_value": p, "cramers_v": cramers_v})
    results_df = pd.DataFrame(results).sort_values("cramers_v", ascending=False)
    return results_df[results_df["cramers_v"] > min_score]

significant_features_df = perform_cramers_v(
    df=unicef_df,
    min_score=0.05,
    target_column_name="FCF26_binary",
    ignore_colums=["FCF26_binary", "FCF26"]

)
print(f"{len(significant_features_df)} features retained")
significant_features_df

29 features retained


,feature_name,p_value,cramers_v
28,wscore,5.303854e-01,0.998658
0,HH1,7.369146e-32,0.366855
10,CL3,9.147104e-03,0.129048
58,LS2,5.400523e-30,0.114343
57,LS1,4.425543e-28,0.104064
27,ethnicity,1.252945e-25,0.102337
80,WS4,2.086852e-02,0.100284
59,LS3,4.184646e-23,0.091711
25,HH7,9.319933e-23,0.088219
44,VT22B,1.747649e-19,0.084207


We remove w-score from our feature set as it represents a measure of child happiness; arguably an alternative target variable rather than a predictor, and its inclusion would risk circular reasoning. We additionally remove ethnicity to prevent the introduction of racial bias into a model that will inform resource allocation decisions. While ethnicity shows a weak but non-negligible association with depression (Cramér's V = 0.1, accounting for ~1% of variance), this association likely reflects proxy effects of socioeconomic status, cultural differences in self-reporting, or experiences of discrimination; factors better captured by other features in the dataset. We note that any future reintroduction of ethnicity would require significant ethical justification. This leaves us with 27 features.

In [30]:
significant_features_df = significant_features_df[~significant_features_df["feature_name"].isin(["ethnicity", "wscore"])]
significant_features: list[str] = significant_features_df["feature_name"].to_list()
significant_features

['HH1',
 'CL3',
 'LS2',
 'LS1',
 'WS4',
 'LS3',
 'HH7',
 'VT22B',
 'MA2',
 'HC4',
 'VT22A',
 'CL13',
 'LS4',
 'WS11',
 'VT22C',
 'HH2',
 'WS7',
 'HC12',
 'MSTATUS',
 'FCD2H',
 'WS1',
 'FCD2J',
 'VT20',
 'HC5',
 'HC8',
 'VT22D',
 'WB4']

We copy our dataframe, keeping only the significant features identified, and the target "FCF26_binary"

In [31]:
unicef_df_filtered = unicef_df[significant_features + ["FCF26_binary"]].copy()

We further look at the percentage of nan values in a column. "CL3" has 68% values nan, "MA2" 23%, "CL13" 15%, "FCD2H" and "FCD2J" 14%, "WS4" 13%.

In [38]:
unicef_df_filtered.isnull().mean().sort_values(ascending=False) * 100

CL3             68.401459
MA2             22.792889
CL13            15.727093
FCD2H           14.420301
FCD2J           14.420301
WS4             13.432609
VT22A            2.537608
LS1              2.537608
VT22B            2.537608
VT22C            2.537608
LS2              2.537608
LS3              2.537608
VT22D            2.537608
WB4              2.537608
VT20             2.537608
LS4              2.537608
FCF26_binary     0.957301
HC4              0.000000
HH1              0.000000
HH7              0.000000
MSTATUS          0.000000
HC12             0.000000
WS7              0.000000
HH2              0.000000
WS11             0.000000
WS1              0.000000
HC5              0.000000
HC8              0.000000
dtype: float64

In [33]:
def identify_columns_of_type_object(df: pd.DataFrame):
    """Identifies the categorical columns that are object data types.
    These columns will need converting into numerical categories such that a model can understand them.
    """
    return df.select_dtypes(include=["object"]).columns.tolist()

categorical_cols = identify_columns_of_type_object(unicef_df_filtered)
print(f"{len(categorical_cols)} categorical object columns found: {categorical_cols}")

23 categorical object columns found: ['LS2', 'LS1', 'WS4', 'LS3', 'HH7', 'VT22B', 'MA2', 'HC4', 'VT22A', 'CL13', 'LS4', 'WS11', 'VT22C', 'WS7', 'HC12', 'MSTATUS', 'FCD2H', 'WS1', 'FCD2J', 'VT20', 'HC5', 'HC8', 'VT22D']
